**Dataset:** Retail_Transaction_Dataset.csv

**Context:** Retail and Consumer Behavior

The objective of this assessment is to organize, explore, and transform transactional data to produce **actionable, reproducible, and technically sound analyses.**
I performed **cohort analysis, time-series exploration, window functions, and textual data processing** to identify trends in consumer behavior and product performance.

**Skills demonstrated:** SQL analysis, cohort analysis, window functions, time-series, text analysis
**Language:** SQL

**Questão 1 - Preparação para análise**

**1.1** Considerando que se trata de um único arquivo CSV, estamos lidando aqui com apenas uma tabela. Nesta podemos identificar algumas colunas, as principais dentre elas e suas relações são:
- "Product ID" e "ProductCategory" - ligam-se ao portfólio de produtos. Podemos analisar qual a categoria que possue mais saída;
- "TransactionDate" - permite análises temporais e sazonais, identificando qual o período onde são realizadas mais compras e menos compras;
- "StoreLocation" -  permite analisar se existem lojas que vendem mais/menos, verificando a quantidade de compras realizadas. 

**1.2** Após análise das faixas de preços podemos detectar que a maior parte dos produtos vendidos está concentrada na faixa de valores entre 50 e 99, 99, somando um total de 55,71%. Podemos notar também que a faixa de valores entre 10 e 49,99 também está bem representada, com um total de 44,27%. 

**1.3** Tendo em conta a questão anterior, os preços foram colocados nas categorias indicadas. 

**1.4** Ao visualizarmos este par de colunas "ProductCategory" e "PaymentMethod" utilizando o PARTITION BY e somando o total de método de pagamento por categoria de produto podemos concluir que existem quatro métodos de pagamento e que estes estão presentes em todas as categorias de forma quase uniforme, não tendo uma preferência por método. 

**1.5** Após verificar que o máximo de quantidades vendidas por CostumerID presente no Dataset é de 9, foram definidas transações de "baixo volume" representando de 1 a 3 itens, típicas de consumo individual; "médio volume" de 4 a 6 itens uma compra intermediária, podendo ser vista como familiar e por último "alto volume" de 7 a 9 itens indicando clientes com maior poder de compra.



In [1]:
df_1 = _dntk.execute_sql(
  'CREATE TABLE Retail_Transaction AS\nSELECT *\nFROM read_csv_auto(\'Retail_Transaction_Dataset.csv\', HEADER=TRUE);\n\nSELECT *\nFROM Retail_Transaction\nLIMIT 10;\n\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_1

,CustomerID,ProductID,Quantity,Price,TransactionDate,PaymentMethod,StoreLocation,ProductCategory,DiscountApplied(%),TotalAmount
0,109318,C,7,80.079844,12/26/2023 12:32,Cash,"176 Andrew Cliffs\nBaileyfort, HI 93354",Books,18.677100,455.862764
1,993229,C,4,75.195229,8/5/2023 0:00,Cash,"11635 William Well Suite 809\nEast Kara, MT 19483",Home Decor,14.121365,258.306546
2,579675,A,8,31.528816,3/11/2024 18:51,Cash,"910 Mendez Ville Suite 909\nPort Lauraland, MO...",Books,15.943701,212.015651
3,799826,D,5,98.880218,10/27/2023 22:00,PayPal,"87522 Sharon Corners Suite 500\nLake Tammy, MO...",Books,6.686337,461.343769
4,121413,A,7,93.188512,12/22/2023 11:38,Cash,"0070 Michelle Island Suite 143\nHoland, VA 80142",Electronics,4.030096,626.030484
5,463050,D,3,54.093152,8/15/2023 4:24,Cash,"8492 Jonathan Drive\nNorth Robertshire, TN 67532",Electronics,10.888768,144.609223
6,888163,D,7,13.121937,12/26/2023 5:32,PayPal,USNV Harrell\nFPO AA 62814,Clothing,16.295127,76.885907
7,843385,A,8,56.025164,10/11/2023 6:48,Debit Card,"489 Juan Loop Apt. 093\nNorth Brettville, WV 7...",Home Decor,6.344306,419.766052
8,839609,B,5,23.857981,2/27/2024 11:13,Credit Card,528 Justin Expressway Apt. 336\nCabreraborough...,Electronics,18.703997,96.977925
9,184135,D,4,63.342777,11/5/2023 1:46,Debit Card,"189 Wright Mews\nMartinfurt, MO 75932",Books,7.640607,234.012018


In [2]:
df_2 = _dntk.execute_sql(
  '-- Questão 1.1 : Abaixo estão indicados os tipos de cada coluna através da instrução "PRAGMA" em cima da tabela \n-- criada no bloco acima. \n\nPRAGMA table_info(\'Retail_Transaction\');',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_2

,cid,name,type,notnull,dflt_value,pk
0,0,CustomerID,BIGINT,False,None,False
1,1,ProductID,VARCHAR,False,None,False
2,2,Quantity,BIGINT,False,None,False
3,3,Price,DOUBLE,False,None,False
4,4,TransactionDate,VARCHAR,False,None,False
5,5,PaymentMethod,VARCHAR,False,None,False
6,6,StoreLocation,VARCHAR,False,None,False
7,7,ProductCategory,VARCHAR,False,None,False
8,8,DiscountApplied(%),DOUBLE,False,None,False
9,9,TotalAmount,DOUBLE,False,None,False


In [3]:
df_3 = _dntk.execute_sql(
  'SELECT \n    CASE\n        WHEN Price < 10 THEN \'0 - 9.99\'\n        WHEN Price BETWEEN 10 AND 49.99 THEN \'Baixo_custo\'\n        WHEN Price BETWEEN 50 AND 99.99 THEN \'Medio_custo\'\n        WHEN Price BETWEEN 100 AND 199.99 THEN \'100 - 199.99\'\n        ELSE \'Alto_custo\'\n    END AS PriceRange, \n    COUNT(*) AS TotalProdutos, \n    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM Retail_Transaction), 2) AS Percentual\nFROM Retail_Transaction\nGROUP BY PriceRange\nORDER BY \n    CASE\n        WHEN PriceRange = \'0 - 9.99\' THEN 1\n        WHEN PriceRange = \'Baixo_custo\' THEN 2\n        WHEN PriceRange = \'Medio_custo\' THEN 3\n        WHEN PriceRange = \'100 - 199.99\' THEN 4\n        WHEN PriceRange = \'Alto_custo\' THEN 5\n    END;',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_3

,PriceRange,TotalProdutos,Percentual
0,Baixo_custo,44266,44.27
1,Medio_custo,55714,55.71
2,Alto_custo,20,0.02


In [4]:
df_4 = _dntk.execute_sql(
  'SELECT\n    ProductCategory, \n    PaymentMethod, \n    COUNT(*) AS TotalTransacoes, \n    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY ProductCategory), 2) AS PercentualPorCategoria\nFROM Retail_Transaction\nGROUP BY ProductCategory, PaymentMethod\nORDER BY ProductCategory, TotalTransacoes DESC;',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_4

,ProductCategory,PaymentMethod,TotalTransacoes,PercentualPorCategoria
0,Books,PayPal,6341,25.33
1,Books,Credit Card,6256,24.99
2,Books,Cash,6243,24.94
3,Books,Debit Card,6191,24.73
4,Clothing,Credit Card,6345,25.32
5,Clothing,PayPal,6285,25.08
6,Clothing,Cash,6278,25.06
7,Clothing,Debit Card,6148,24.54
8,Electronics,Credit Card,6295,25.19
9,Electronics,Cash,6282,25.14


In [5]:
df_5 = _dntk.execute_sql(
  'SELECT MAX(Quantity) AS Max_Quantity\nFROM Retail_Transaction;\n\nSELECT\n    CustomerID,\n    ProductID,\n    Quantity,\n    CASE\n        WHEN Quantity BETWEEN 1 AND 3 THEN \'Baixo Volume\'\n        WHEN Quantity BETWEEN 4 AND 6 THEN \'Médio Volume\'\n        WHEN Quantity BETWEEN 7 AND 9 THEN \'Alto Volume\'\n        ELSE \'Indefinido\'\n    END AS VolumeCategoria,\n    Price,\n    TotalAmount\nFROM Retail_Transaction\nLIMIT 20;',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_5

,CustomerID,ProductID,Quantity,VolumeCategoria,Price,TotalAmount
0,109318,C,7,Alto Volume,80.079844,455.862764
1,993229,C,4,Médio Volume,75.195229,258.306546
2,579675,A,8,Alto Volume,31.528816,212.015651
3,799826,D,5,Médio Volume,98.880218,461.343769
4,121413,A,7,Alto Volume,93.188512,626.030484
5,463050,D,3,Baixo Volume,54.093152,144.609223
6,888163,D,7,Alto Volume,13.121937,76.885907
7,843385,A,8,Alto Volume,56.025164,419.766052
8,839609,B,5,Médio Volume,23.857981,96.977925
9,184135,D,4,Médio Volume,63.342777,234.012018


**Questão 2 - Padrões com funções de janela**

**2.1** Considerando que nese Dataset não existe uma coluna que indique o país e/ou o estado explicitamente, a análise foi feita em cima da coluna "CustomerID" e "ProductCategory".

**2.2** Através deste código e do seu resultado podemos visualizar quais clientes fizeram mais que uma compra e o seu engajamento, dependendo se a média está aumentando ou diminuindo conforme suas mais recentes compras. 

**2.3** Considerando que nese Dataset não existe uma coluna que indique o país e/ou o estado explicitamente, a análise foi feita em cima da coluna "TotalAmount" e "ProductCategory". Analisando o dataframe criado, podemos ver quais os clientes que mais consumiram em cada categoria de produto. 

**2.4** Neste código foi definida uma janela reutilizável cuja expressão "w" foi utilizada em uma consulta com RANK() e SUM() para identificar os clientes que mais consumiram de cada categoria.  

**2.5** Neste código foi utilizada a função WINDOW com a expressão "w" para deixar o código mais limpo e o CASE WHEN com o LAG() para classificar a relação compra atual x compra anterior como "Primeira Compra", "Redução" ou "Crescimento" para cada CustomerID. Pelo fato de a grande maioria dos clientes fazerem apenas uma compra, a classificação (quase que de forma homogênea) fica como "Primeira Compra".


In [6]:
df_6 = _dntk.execute_sql(
  'SELECT\n    CustomerID,\n    ProductCategory,\n    COUNT(*) AS TotalPedidos,\n    SUM(TotalAmount) AS ValorTotalGasto,\n    SUM(Quantity) AS ItensTotais\nFROM Retail_Transaction\nGROUP BY CustomerID, ProductCategory\nORDER BY ValorTotalGasto DESC\nLIMIT 20;',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_6

,CustomerID,ProductCategory,TotalPedidos,ValorTotalGasto,ItensTotais
0,903169,Books,2,1563.851160,17.0
1,780013,Clothing,2,1519.193458,17.0
2,823783,Clothing,3,1441.329882,22.0
3,887487,Home Decor,2,1352.743062,16.0
4,598581,Home Decor,2,1342.070296,17.0
5,536978,Clothing,2,1327.266128,15.0
6,400993,Electronics,2,1314.806049,18.0
7,32895,Electronics,2,1291.913188,15.0
8,895452,Electronics,2,1241.740674,15.0
9,465262,Electronics,2,1222.491943,18.0


In [7]:
df_7 = _dntk.execute_sql(
  'CREATE TABLE Retail_Transaction_Clean AS\nSELECT\n    CustomerID,\n    ProductID,\n    Quantity,\n    Price,\n    MAKE_TIMESTAMP(\n        CAST(SPLIT_PART(SPLIT_PART(TransactionDate, \' \', 1), \'/\', 3) AS INTEGER), -- Ano\n        CAST(SPLIT_PART(SPLIT_PART(TransactionDate, \' \', 1), \'/\', 1) AS INTEGER), -- Mês\n        CAST(SPLIT_PART(SPLIT_PART(TransactionDate, \' \', 1), \'/\', 2) AS INTEGER), -- Dia\n        CAST(SPLIT_PART(SPLIT_PART(TransactionDate, \' \', 2), \':\', 1) AS INTEGER), -- Hora\n        CAST(SPLIT_PART(SPLIT_PART(TransactionDate, \' \', 2), \':\', 2) AS INTEGER), -- Minuto\n        0 -- Segundos\n    ) AS TransactionDate,\n    PaymentMethod,\n    StoreLocation,\n    ProductCategory,\n    \'DiscountApplied(%)\',\n    TotalAmount\nFROM Retail_Transaction;\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_7

,Count
0,100000


In [8]:
df_8 = _dntk.execute_sql(
  'SELECT\n    CustomerID, \n    TransactionDate, \n    TotalAmount, \n    ROUND(\n        AVG(TotalAmount) OVER (\n            PARTITION BY CustomerID\n            ORDER BY TransactionDate\n            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW\n        ), 2\n    ) AS MediaMovel3Compras\nFROM Retail_Transaction_Clean\nORDER BY CustomerID, TransactionDate;',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_8

,CustomerID,TransactionDate,TotalAmount,MediaMovel3Compras
0,14,2023-08-06 06:45:00,256.232791,256.23
1,42,2023-05-19 21:52:00,502.656523,502.66
2,49,2023-06-05 13:10:00,21.399047,21.40
3,59,2023-08-19 03:50:00,139.612036,139.61
4,59,2024-04-01 01:06:00,109.880660,124.75
...,...,...,...,...
99995,999910,2023-10-13 08:22:00,12.441084,12.44
99996,999931,2023-09-04 15:47:00,105.039745,105.04
99997,999977,2023-07-03 10:19:00,71.135444,71.14
99998,999990,2023-08-26 10:48:00,195.449291,195.45


In [9]:
df_9 = _dntk.execute_sql(
  'SELECT\n    ProductCategory, \n    CustomerID, \n    SUM(TotalAmount) AS ValorTotalGasto, \n    RANK() OVER(\n        PARTITION BY ProductCategory\n        ORDER BY SUM(TotalAmount) DESC\n    ) AS PosicaoRanking\nFROM Retail_Transaction_Clean\nGROUP BY ProductCategory, CustomerID\nQUALIFY PosicaoRanking <= 3\nORDER BY ProductCategory, PosicaoRanking;\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_9

,ProductCategory,CustomerID,ValorTotalGasto,PosicaoRanking
0,Books,903169,1563.851160,1
1,Books,326398,1185.374449,2
2,Books,392763,1165.913585,3
3,Clothing,780013,1519.193458,1
4,Clothing,823783,1441.329882,2
5,Clothing,536978,1327.266128,3
6,Electronics,400993,1314.806049,1
7,Electronics,32895,1291.913188,2
8,Electronics,895452,1241.740674,3
9,Home Decor,887487,1352.743062,1


In [10]:
df_10 = _dntk.execute_sql(
  'WITH Totais AS (\n    SELECT\n        CustomerID,\n        ProductCategory,\n        SUM(TotalAmount) AS ValorTotalGasto\n    FROM Retail_Transaction_Clean\n    GROUP BY CustomerID, ProductCategory\n)\nSELECT\n    CustomerID,\n    ProductCategory,\n    ValorTotalGasto,\n    RANK() OVER w AS RankingCliente,\n    SUM(ValorTotalGasto) OVER w AS SomaCategoria,\n    ROUND(\n        100.0 * ValorTotalGasto / SUM(ValorTotalGasto) OVER (PARTITION BY ProductCategory), \n        2\n    ) AS PercentualCategoria\nFROM Totais\nWINDOW w AS (\n    PARTITION BY ProductCategory\n    ORDER BY ValorTotalGasto DESC\n)\nORDER BY ProductCategory, RankingCliente;',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_10

,CustomerID,ProductCategory,ValorTotalGasto,RankingCliente,SomaCategoria,PercentualCategoria
0,903169,Books,1563.851160,1,1.563851e+03,0.02
1,326398,Books,1185.374449,2,2.749226e+03,0.02
2,392763,Books,1165.913585,3,3.915139e+03,0.02
3,39999,Books,1148.629689,4,5.063769e+03,0.02
4,980072,Books,1136.449742,5,6.200219e+03,0.02
...,...,...,...,...,...,...
98805,987741,Home Decor,8.687648,24634,6.173387e+06,0.00
98806,317290,Home Decor,8.677368,24635,6.173396e+06,0.00
98807,42314,Home Decor,8.627679,24636,6.173405e+06,0.00
98808,528749,Home Decor,8.464777,24637,6.173413e+06,0.00


In [11]:
df_11 = _dntk.execute_sql(
  'SELECT\n    CustomerID, \n    TransactionDate, \n    TotalAmount, \n    LAG(TotalAmount) OVER w AS CompraAnterior, \n    CASE\n        WHEN LAG(TotalAmount) OVER w IS NULL\n            THEN \'Primeira Compra\'\n        WHEN TotalAmount > LAG(TotalAmount) OVER w\n            THEN \'Crescimento\'\n        WHEN TotalAmount < LAG(TotalAmount) OVER w\n            THEN \'Redução\'\n        ELSE \'Estável\'\n    END AS Tendencia\nFROM Retail_Transaction_Clean\nWINDOW w AS (\n    PARTITION BY CustomerID\n    ORDER BY TransactionDate\n)\nORDER BY CustomerID, TransactionDate\nLIMIT 50;\n\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_11

,CustomerID,TransactionDate,TotalAmount,CompraAnterior,Tendencia
0,14,2023-08-06 06:45:00,256.232791,NaN,Primeira Compra
1,42,2023-05-19 21:52:00,502.656523,NaN,Primeira Compra
2,49,2023-06-05 13:10:00,21.399047,NaN,Primeira Compra
3,59,2023-08-19 03:50:00,139.612036,NaN,Primeira Compra
4,59,2024-04-01 01:06:00,109.880660,139.612036,Redução
5,65,2023-06-18 10:29:00,548.006625,NaN,Primeira Compra
6,87,2023-08-21 15:23:00,41.515905,NaN,Primeira Compra
7,96,2024-04-06 12:35:00,194.356816,NaN,Primeira Compra
8,98,2023-12-20 11:49:00,166.934647,NaN,Primeira Compra
9,100,2023-05-13 17:17:00,710.062576,NaN,Primeira Compra


**Questão 3 - Séries temporais**

**3.1** Análise de informações temporais a partir do EXTRACT() feito em cima dos meses, dias da semana e hora:
- Meses: podemos observar que o mês com menos pedidos e menos receita foi Abril, enquanto o com mais pedidos foi Junho.
- Dia da Semana: considerando que o DuckDB contabiliza os dias da semana de 0 a 6, sendo 0=Domingo e 6=Sábado, podemos ver que o dia da semana com mais saída de produtos/maior lucro é sábado e o dia com menos saída/maior lucro é terça-feira.
- Hora: a hora com mais saída de pedidos é 19h, entretanto com mais lucro é 17h, indicando que neste último os valores dos produtos e/ou a quantidade de produtos por pedido foi maior do que das 19h.

**3.2** A média de dias entre compras consecutivas por cada cliente varia de 0 a 343 days. A partir desse insight, podemos fazer futuras análises para descobrir em quais dias exatos os clientes compraram, se há um mês que tais clientes compraram mais, quantas compras foram realizadas em um todo por cada cliente e dessa forma gerar estratégias de alavancagem por intermédio de campanhas de marketing.

**3.3** Considerando que temos somente esta tabela como referência, a análise foi feita em cima das colunas "TotalAmount", "Quantity" e foi criada uma terceira coluna "DisccountApplied_Derived" para verificar se foi aplicado algum desconto ou não em cima do valor total. Esta alternativa foi criada devido ao problema com o nome da coluna x DeepNote (que não interpreta o % corretamente). A linha do tempo foi aplicada para cada Customer da tabela. 

**3.4** A partir desta query, podemos verificar que há algumas semanas com quantidade de pedidos acima de 2000 como: 2024-01-29, 2023-10-30, 2023-07-31, o que pode indicar alguma promoção, campanha ou até mesmo mudança de salário/comportamento como 13 salário, férias... Além disso podemos notar que estas datas caem no final do mês, o que coincide com a data de pagamento de muitas empresas. Através desta análise, também podemos notar que há uma dinâmica regular de pedidos feitos, que rondam os 1800-1900, com exceção da primeira semana, com um total de 294 pedidos feitos. 

**3.5** Esta query complementa a última, tendo as maiores variações percentuais relacionadas à primeira semana e as datas 2024-01-29, 2023-10-30, 2023-07-31. Este tipo de análise é importante para saber como as vendas estão acontecendo, qual a margem de lucro e se há muita variação entre uma semana e outra. 

**3.6** A aplicação desta janela deslizante permite comparações semana a semana e reduz o impacto de variações pontuais causadas por promoções ou datas específicas. 

**3.7** Através desta query podemos ter um apanhado geral do volume de vendas por mês e sua influencia no valor total de um período de um ano a partir da data de início , 2023/04. 

**3.8** Analisando o resultado podemos concluir que não há muita variação de pedidos/receita de um mês para o outro. Como o dataset é limitado ao período de um ano, não temos a possibilidade de comparar estes mesmos meses em outros anos. 
Esta análise quando aplicada em diferentes anos, pode levar a conclusões sobre a própria empresa e a estratégia que está sendo utilizada, se está funcionando ou se deve ser estudada uma nova estratégia. 

In [12]:
df_12 = _dntk.execute_sql(
  'SELECT\n    CustomerID, \n    TransactionDate, \n    EXTRACT(MONTH FROM TransactionDate) AS Mes, \n    EXTRACT(DAYOFWEEK FROM TransactionDate) AS DiaSemana, \n    EXTRACT(HOUR FROM TransactionDate) AS Hora\nFROM Retail_Transaction_Clean\nLIMIT 20;\n\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_12

,CustomerID,TransactionDate,Mes,DiaSemana,Hora
0,109318,2023-12-26 12:32:00,12,2,12
1,993229,2023-08-05 00:00:00,8,6,0
2,579675,2024-03-11 18:51:00,3,1,18
3,799826,2023-10-27 22:00:00,10,5,22
4,121413,2023-12-22 11:38:00,12,5,11
5,463050,2023-08-15 04:24:00,8,2,4
6,888163,2023-12-26 05:32:00,12,2,5
7,843385,2023-10-11 06:48:00,10,3,6
8,839609,2024-02-27 11:13:00,2,2,11
9,184135,2023-11-05 01:46:00,11,0,1


In [13]:
df_13 = _dntk.execute_sql(
  'SELECT\n    EXTRACT(HOUR FROM TransactionDate) AS Hora, \n    COUNT(*) AS TotalPedidos, \n    SUM(TotalAmount) AS ReceitaTotal\nFROM Retail_Transaction_Clean\nGROUP BY Hora\nORDER BY Hora;\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_13

,Hora,TotalPedidos,ReceitaTotal
0,0,4118,1.043571e+06
1,1,4247,1.067015e+06
2,2,4135,1.036085e+06
3,3,4147,1.028816e+06
4,4,4226,1.036635e+06
5,5,4198,1.053061e+06
6,6,4163,1.041149e+06
7,7,4231,1.051865e+06
8,8,4172,1.031465e+06
9,9,4238,1.053650e+06


In [14]:
df_14 = _dntk.execute_sql(
  'SELECT\n    EXTRACT(MONTH FROM TransactionDate) AS Mes, \n    COUNT(*) AS TotalPedidos, \n    SUM(TotalAmount) AS ReceitaTotal\nFROM Retail_Transaction_Clean\nGROUP BY Mes\nORDER BY Mes;',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_14

,Mes,TotalPedidos,ReceitaTotal
0,1,8543,2.128345e+06
1,2,8076,1.973154e+06
2,3,8457,2.108248e+06
3,4,7921,1.939190e+06
4,5,8388,2.099576e+06
5,6,8243,2.066365e+06
6,7,8597,2.132551e+06
7,8,8498,2.109353e+06
8,9,8181,2.050335e+06
9,10,8325,2.049451e+06


In [15]:
df_15 = _dntk.execute_sql(
  'SELECT\n    EXTRACT(DAYOFWEEK FROM TransactionDate) AS DiaSemana, \n    COUNT(*) AS TotalPedidos, \n    SUM(TotalAmount) AS ReceitaTotal\nFROM Retail_Transaction_Clean\nGROUP BY DiaSemana\nORDER BY DiaSemana;',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_15

,DiaSemana,TotalPedidos,ReceitaTotal
0,0,14291,3.535350e+06
1,1,14259,3.493463e+06
2,2,14212,3.529848e+06
3,3,14288,3.540645e+06
4,4,14344,3.604946e+06
5,5,14252,3.547942e+06
6,6,14354,3.581302e+06


In [16]:
df_16 = _dntk.execute_sql(
  'WITH Diferencas AS (\n    SELECT\n        CustomerID, \n        TransactionDate, \n        LAG(TransactionDate) OVER(\n            PARTITION BY CustomerID\n            ORDER BY TransactionDate\n        ) AS CompraAnterior\n    FROM Retail_Transaction_Clean\n)\nSELECT\n    CustomerID, \n    ROUND(AVG(DATEDIFF(\'day\', CompraAnterior, TransactionDate)), 2) AS MediaDiasEntreCompras\nFROM Diferencas\nWHERE CompraAnterior IS NOT NULL\nGROUP BY CustomerID\nORDER BY MediaDiasEntreCompras DESC;',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_16

,CustomerID,MediaDiasEntreCompras
0,778348,363.0
1,784521,360.0
2,641185,356.0
3,464486,351.0
4,327183,351.0
...,...,...
4616,769224,0.0
4617,371801,0.0
4618,965303,0.0
4619,555908,0.0


In [17]:
df_17 = _dntk.execute_sql(
  'WITH base AS (\n  SELECT\n    CustomerID,\n    TransactionDate,\n    TotalAmount,\n    Quantity,\n    Price,\n    CASE\n      WHEN Quantity * Price IS NULL OR Quantity * Price = 0 THEN NULL\n      ELSE 100.0 * (1 - TotalAmount / (Quantity * Price))\n    END AS DiscountApplied_Derived\n  FROM Retail_Transaction_Clean\n)\n\nSELECT\n  CustomerID,\n  TransactionDate,\n  \'Compra\' AS TipoInteracao,\n  TotalAmount\nFROM base\n\nUNION ALL\n\nSELECT\n  CustomerID,\n  TransactionDate,\n  \'Compra com Desconto\' AS TipoInteracao,\n  TotalAmount\nFROM base\n\nUNION ALL\n\nSELECT\n  CustomerID,\n  TransactionDate,\n  \'Grande Volume de Itens\' AS TipoInteracao,\n  TotalAmount\nFROM base\nWHERE Quantity >= 7\n\nORDER BY CustomerID, TransactionDate;',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_17

,CustomerID,TransactionDate,TipoInteracao,TotalAmount
0,14,2023-08-06 06:45:00,Compra,256.232791
1,14,2023-08-06 06:45:00,Compra com Desconto,256.232791
2,42,2023-05-19 21:52:00,Compra,502.656523
3,42,2023-05-19 21:52:00,Compra com Desconto,502.656523
4,42,2023-05-19 21:52:00,Grande Volume de Itens,502.656523
...,...,...,...,...
233450,999990,2023-08-26 10:48:00,Compra,195.449291
233451,999990,2023-08-26 10:48:00,Compra com Desconto,195.449291
233452,999990,2023-08-26 10:48:00,Grande Volume de Itens,195.449291
233453,999997,2023-08-06 07:25:00,Compra,75.614248


In [18]:
df_18 = _dntk.execute_sql(
  'SELECT\n    DATE_TRUNC(\'week\', TransactionDate) AS Semana, \n    COUNT(*) AS TotalPedidos, \n    SUM(TotalAmount) AS ReceitaTotal\nFROM Retail_Transaction_Clean\nGROUP BY 1\nORDER BY Semana;',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_18

,Semana,TotalPedidos,ReceitaTotal
0,2023-04-24,264,60840.787854
1,2023-05-01,1931,481984.084105
2,2023-05-08,1885,476509.160280
3,2023-05-15,1869,458240.848777
4,2023-05-22,1914,481252.924503
5,2023-05-29,1902,479776.144245
6,2023-06-05,1934,474594.408139
7,2023-06-12,1913,482094.301263
8,2023-06-19,1856,471040.740195
9,2023-06-26,1960,490290.312243


In [19]:
df_19 = _dntk.execute_sql(
  'WITH VendasSemanais AS (\n    SELECT\n        DATE_TRUNC(\'week\', TransactionDate) AS Semana, \n        SUM(TotalAmount) AS ReceitaTotal\n    FROM Retail_Transaction_Clean\n    GROUP BY DATE_TRUNC(\'week\', TransactionDate)\n)\nSELECT\n    Semana, \n    ReceitaTotal, \n    LAG(ReceitaTotal) OVER (ORDER BY Semana) AS ReceitaAnterior, \n    ROUND(\n        100.0 * (ReceitaTotal - LAG(ReceitaTotal) OVER (ORDER BY Semana))\n        / NULLIF(LAG(ReceitaTotal) OVER (ORDER BY Semana), 0), \n        2\n    ) AS VariacaoPercentual\nFROM VendasSemanais\nORDER BY Semana;',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_19

,Semana,ReceitaTotal,ReceitaAnterior,VariacaoPercentual
0,2023-04-24,60840.787854,NaN,NaN
1,2023-05-01,481984.084105,60840.787854,692.21
2,2023-05-08,476509.160280,481984.084105,-1.14
3,2023-05-15,458240.848777,476509.160280,-3.83
4,2023-05-22,481252.924503,458240.848777,5.02
5,2023-05-29,479776.144245,481252.924503,-0.31
6,2023-06-05,474594.408139,479776.144245,-1.08
7,2023-06-12,482094.301263,474594.408139,1.58
8,2023-06-19,471040.740195,482094.301263,-2.29
9,2023-06-26,490290.312243,471040.740195,4.09


In [20]:
df_20 = _dntk.execute_sql(
  'WITH VendasDiarias AS (\n    SELECT\n        DATE_TRUNC(\'day\', TransactionDate) AS Dia, \n        SUM(TotalAmount) AS ReceitaTotal\n    FROM Retail_Transaction_Clean\n    GROUP BY DATE_TRUNC(\'day\', TransactionDate)\n)\nSELECT\n    Dia, \n    ReceitaTotal, \n    ROUND(\n        AVG(ReceitaTotal) OVER(\n            ORDER BY Dia\n            RANGE BETWEEN INTERVAL 6 DAY PRECEDING AND CURRENT ROW\n        ), \n        2\n    ) AS MediaMovel7d\nFROM VendasDiarias\nORDER BY Dia;\n    \n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_20

,Dia,ReceitaTotal,MediaMovel7d
0,2023-04-29,2241.869279,2241.87
1,2023-04-30,58598.918575,30420.39
2,2023-05-01,62845.530099,41228.77
3,2023-05-02,76296.207253,49995.63
4,2023-05-03,63935.346826,52783.57
...,...,...,...
361,2024-04-24,67860.159728,67644.35
362,2024-04-25,66335.834049,67562.27
363,2024-04-26,70107.702288,67080.98
364,2024-04-27,60887.308483,67074.55


In [21]:
df_21 = _dntk.execute_sql(
  'WITH VendasMensais AS (\n    SELECT\n        DATE_TRUNC(\'month\', TransactionDate) AS Mes,\n        SUM(TotalAmount) AS ReceitaMensal\n    FROM Retail_Transaction_Clean\n    GROUP BY DATE_TRUNC(\'month\', TransactionDate)\n)\nSELECT\n    Mes,\n    ReceitaMensal,\n    SUM(ReceitaMensal) OVER (\n        ORDER BY Mes\n        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW\n    ) AS ReceitaAcumulada\nFROM VendasMensais\nORDER BY Mes;',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_21

,Mes,ReceitaMensal,ReceitaAcumulada
0,2023-04-01,6.084079e+04,6.084079e+04
1,2023-05-01,2.099576e+06,2.160417e+06
2,2023-06-01,2.066365e+06,4.226782e+06
3,2023-07-01,2.132551e+06,6.359332e+06
4,2023-08-01,2.109353e+06,8.468685e+06
5,2023-09-01,2.050335e+06,1.051902e+07
6,2023-10-01,2.049451e+06,1.256847e+07
7,2023-11-01,2.051277e+06,1.461975e+07
8,2023-12-01,2.125651e+06,1.674540e+07
9,2024-01-01,2.128345e+06,1.887374e+07


In [22]:
df_22 = _dntk.execute_sql(
  'WITH VendasMensais AS (\n    SELECT\n        EXTRACT(MONTH FROM TransactionDate) AS Mes,\n        SUM(TotalAmount) AS ReceitaMensal,\n        COUNT(*) AS Pedidos\n    FROM Retail_Transaction_Clean\n    WHERE TransactionDate BETWEEN \'2023-11-01\' AND \'2023-12-31\'\n    GROUP BY Mes\n)\nSELECT\n    SUM(CASE WHEN Mes = 11 THEN ReceitaMensal ELSE 0 END) AS ReceitaNovembro,\n    SUM(CASE WHEN Mes = 12 THEN ReceitaMensal ELSE 0 END) AS ReceitaDezembro,\n    SUM(CASE WHEN Mes = 11 THEN Pedidos ELSE 0 END) AS PedidosNovembro,\n    SUM(CASE WHEN Mes = 12 THEN Pedidos ELSE 0 END) AS PedidosDezembro\nFROM VendasMensais;',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_22

,ReceitaNovembro,ReceitaDezembro,PedidosNovembro,PedidosDezembro
0,2.051277e+06,2.059765e+06,8298.0,8206.0


**Questão 4 - Cohort Analysis**

**4.1** Essa análise permite observar quantos clientes novos entraram em cada mês e assim avaliar a evolução da aquisição de clientes ao longo do ano. Neste dataframe por exemplo, podemos verificar que o mês de Julho foi o mais forte, com um total de 8436 que fizeram sua primeira compra e o segundo mês mais fraco (tirando 2023/04) foi o último mês avaliado, com um total de 6935.

**4.2** Este tipo de análise permite entender a taxa de retenção dos clientes; em quanto tempo da primeira compra retornam a comprar; é possível até fazer uma relação com a categoria do produto que foi inicialmente comprado x a categoria do produto da compra atual; podemos verificar se a retenção tem a ver com a altura do ano em que foi feita a primeira compra ou não. 

**4.3** Nesta análise, com a coluna extra contabilizando a porcentagem de taxa de sobrevivência, ou seja, dos Customers que continuaram comprando após a primeira compra, podemos ver que a quantidade é muito pequena, abaixo de 1%, o que indica que a grande maioria contempla-se apenas com uma única compra, o que pode ser um problema para a empresa.

**4.4** Através dessa análise podemos observar que a grande maioria dos clientes que voltam a comprar (91.3%) o fazem em um período de 30 dias ou mais, o que representa para a empresa um alto risco de *churn*.

**4.5** Através da comparação entre as coortes podemos identificar se há uma diferença na velocidade com que os clientes fazem suas compras nos seus primeiros meses. Além disso podemos acompanhar a evolução dentro da própria coorte, conforme a receita mensal vai sendo somada devido ao janelamento embutido no código. 

**4.6** Nesta análise onde o foco está nos meses '2023-04-01' e '2023-10-01' fica clara a disparidade de receita acumulada no decorrer dos meses. De um lado temos o acumulado de receita considerando os clientes que aderiram no mês de abril cujas próximas compras foram espalhadas nos seguintes 11 meses, totalizando 66.899, 44925 moedas e de outro lado o acumulado de receita considerando os clientes que aderiram no mês de outubro, cujas próximas compras foram espalhadas nos seguintes 5 meses, totalizando 2.058.684,663 moedas.


In [23]:
df_23 = _dntk.execute_sql(
  'WITH PrimeiraCompra AS (\n    SELECT\n        CustomerID, \n        MIN(DATE_TRUNC(\'month\', TransactionDate)) AS MesEntrada\n    FROM Retail_Transaction_Clean\n    GROUP BY CustomerID\n)\nSELECT\n    p.CustomerID, \n    p.MesEntrada\nFROM PrimeiraCompra p\nORDER BY p.MesEntrada, p.CustomerID;\n\nWITH PrimeiraCompra AS (\n    SELECT\n        CustomerID, \n        MIN(DATE_TRUNC(\'month\', TransactionDate)) AS MesEntrada\n    FROM Retail_Transaction_Clean\n    GROUP BY CustomerID\n)\nSELECT\n    MesEntrada, \n    COUNT(DISTINCT CustomerID) AS TotalClientesCoorte\nFROM PrimeiraCompra\nGROUP BY MesEntrada\nORDER BY MesEntrada;\n\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_23

,MesEntrada,TotalClientesCoorte
0,2023-04-01,264
1,2023-05-01,8330
2,2023-06-01,8149
3,2023-07-01,8436
4,2023-08-01,8260
5,2023-09-01,7903
6,2023-10-01,7941
7,2023-11-01,7843
8,2023-12-01,7985
9,2024-01-01,7966


In [24]:
df_24 = _dntk.execute_sql(
  'WITH PrimeiraCompra AS (\n    SELECT\n        CustomerID, \n        MIN(DATE_TRUNC(\'month\', TransactionDate)) AS MesEntrada\n    FROM Retail_Transaction_Clean\n    GROUP BY CustomerID\n), \nComprasComCoorte AS (\n    SELECT\n        r.CustomerID, \n        DATE_TRUNC(\'month\', r.TransactionDate) AS MesCompra, \n        p.MesEntrada, \n        DATE_DIFF(\'month\', p.MesEntrada, DATE_TRUNC(\'month\', r.TransactionDate)) AS MesRelativo\n    FROM Retail_Transaction_Clean r\n    JOIN PrimeiraCompra p USING(CustomerID)\n)\nSELECT\n    MesEntrada, \n    MesRelativo, \n    COUNT(DISTINCT CustomerID) AS ClientesAtivos\nFROM ComprasComCoorte\nGROUP BY MesEntrada, MesRelativo\nORDER BY MesEntrada, MesRelativo;',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_24

,MesEntrada,MesRelativo,ClientesAtivos
0,2023-04-01,0,264
1,2023-04-01,1,3
2,2023-04-01,2,1
3,2023-04-01,4,3
4,2023-04-01,5,3
...,...,...,...
83,2024-02-01,1,57
84,2024-02-01,2,68
85,2024-03-01,0,7731
86,2024-03-01,1,71


In [25]:
df_25 = _dntk.execute_sql(
  'WITH PrimeiraCompra AS( \n    SELECT\n        CustomerID, \n        MIN(DATE_TRUNC(\'month\', TransactionDate)) AS MesEntrada\n    FROM Retail_Transaction_Clean\n    GROUP BY CustomerID\n), \nComprasComCoorte AS(\n    SELECT\n        r.CustomerID, \n        DATE_TRUNC(\'month\', r.TransactionDate) AS MesCompra, \n        p.MesEntrada, \n        DATE_DIFF(\'month\', p.MesEntrada, DATE_TRUNC(\'month\', r.TransactionDate)) AS MesRelativo\n    FROM Retail_Transaction_Clean r\n    JOIN PrimeiraCompra p USING (CustomerID)\n), \nAtivosPorCoorte AS (\n    SELECT\n        MesEntrada, \n        MesRelativo, \n        COUNT(DISTINCT CustomerID) AS ClientesAtivos\n    FROM ComprasComCoorte\n    GROUP BY MesEntrada, MesRelativo\n), \nTamanhoCoorte AS (\n    SELECT\n        MesEntrada, \n        ClientesAtivos AS ClientesIniciais\n    FROM AtivosPorCoorte\n    WHERE MesRelativo = 0\n)\nSELECT\n    a.MesEntrada, \n    a.MesRelativo, \n    a.ClientesAtivos, \n    ROUND(100.0 * a.ClientesAtivos / t.ClientesIniciais, 2) AS TaxaSobrevivencia\nFROM AtivosPorCoorte a \nJOIN TamanhoCoorte t USING (MesEntrada)\nORDER BY a.MesEntrada, a.MesRelativo;',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_25

,MesEntrada,MesRelativo,ClientesAtivos,TaxaSobrevivencia
0,2023-04-01,0,264,100.00
1,2023-04-01,1,3,1.14
2,2023-04-01,2,1,0.38
3,2023-04-01,4,3,1.14
4,2023-04-01,5,3,1.14
...,...,...,...,...
83,2024-02-01,1,57,0.76
84,2024-02-01,2,68,0.91
85,2024-03-01,0,7731,100.00
86,2024-03-01,1,71,0.92


In [26]:
df_26 = _dntk.execute_sql(
  'WITH ComprasOrdenadas AS (\n    SELECT\n        CustomerID, \n        TransactionDate, \n        LAG(TransactionDate) OVER (\n            PARTITION BY CustomerID\n            ORDER BY TransactionDate\n        ) AS CompraAnterior\n    FROM Retail_Transaction_Clean\n)\nSELECT\n    CustomerID, \n    TransactionDate AS DataReativacao, \n    CompraAnterior, \n    DATEDIFF(\'day\', CompraAnterior, TransactionDate) AS DiasInatividade, \n    CASE\n        WHEN DATEDIFF(\'day\', CompraAnterior, TransactionDate) >= 30 THEN \'Reativado 30+ dias\'\n        WHEN DATEDIFF(\'day\', CompraAnterior, TransactionDate) >= 15 THEN \'Reativado 15+ dias\'\n        ELSE NULL\n    END AS StatusReativacao\nFROM ComprasOrdenadas\nWHERE CompraAnterior IS NOT NULL\n    AND DATEDIFF(\'day\', CompraAnterior, TransactionDate) >= 15\nORDER BY CustomerID, TransactionDate;\n ',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_26

,CustomerID,DataReativacao,CompraAnterior,DiasInatividade,StatusReativacao
0,59,2024-04-01 01:06:00,2023-08-19 03:50:00,226,Reativado 30+ dias
1,299,2024-01-06 14:32:00,2023-12-06 18:02:00,31,Reativado 30+ dias
2,442,2024-03-20 08:21:00,2023-08-30 03:37:00,203,Reativado 30+ dias
3,959,2023-08-03 23:54:00,2023-07-06 08:28:00,28,Reativado 15+ dias
4,1614,2023-05-25 01:40:00,2023-05-10 18:14:00,15,Reativado 15+ dias
...,...,...,...,...,...
4390,998832,2024-01-09 18:22:00,2023-07-03 07:18:00,190,Reativado 30+ dias
4391,999390,2023-10-23 06:36:00,2023-06-19 16:23:00,126,Reativado 30+ dias
4392,999509,2023-12-08 18:54:00,2023-11-18 12:27:00,20,Reativado 15+ dias
4393,999766,2024-03-05 14:29:00,2024-01-29 08:51:00,36,Reativado 30+ dias


In [27]:
df_27 = _dntk.execute_sql(
  'WITH PrimeiraCompra AS (\n    SELECT\n        CustomerID, \n        MIN(DATE_TRUNC(\'month\', TransactionDate)) AS MesEntrada\n    FROM Retail_Transaction_Clean\n    GROUP BY CustomerID\n), \nComprasComCoorte AS(\n    SELECT\n        r.CustomerID, \n        DATE_TRUNC(\'month\', r.TransactionDate) AS MesCompra, \n        p.MesEntrada, \n        DATE_DIFF(\'month\', p.MesEntrada, DATE_TRUNC(\'month\', r.TransactionDate)) AS MesRelativo, \n        r.TotalAmount\n    FROM Retail_Transaction_Clean r\n    JOIN PrimeiraCompra p USING (CustomerID)\n)\nSELECT\n    MesEntrada, \n    MesRelativo, \n    SUM(TotalAmount) AS ReceitaMes, \n    SUM(SUM(TotalAmount)) OVER (\n        PARTITION BY MesEntrada\n        ORDER BY MesRelativo\n        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW\n    ) AS ReceitaAcumulada\nFROM ComprasComCoorte\nWHERE MesRelativo BETWEEN 0 AND 5\nGROUP BY MesEntrada, MesRelativo\nORDER BY MesEntrada, MesRelativo;',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_27

,MesEntrada,MesRelativo,ReceitaMes,ReceitaAcumulada
0,2023-04-01,0,6.084079e+04,6.084079e+04
1,2023-04-01,1,6.192800e+02,6.146007e+04
2,2023-04-01,2,6.505560e+01,6.152512e+04
3,2023-04-01,4,6.709938e+02,6.219612e+04
4,2023-04-01,5,4.194533e+02,6.261557e+04
...,...,...,...,...
57,2024-02-01,1,1.556813e+04,1.852475e+06
58,2024-02-01,2,1.801455e+04,1.870490e+06
59,2024-03-01,0,1.929012e+06,1.929012e+06
60,2024-03-01,1,1.959129e+04,1.948603e+06


In [28]:
df_28 = _dntk.execute_sql(
  'WITH PrimeiraCompra AS (\n    SELECT\n        CustomerID, \n        MIN(DATE_TRUNC(\'month\', TransactionDate)) AS MesEntrada\n    FROM Retail_Transaction_Clean\n    GROUP BY CustomerID\n), \nComprasComCoorte AS (\n    SELECT\n        r.CustomerID, \n        DATE_TRUNC(\'month\', r.TransactionDate) AS MesCompra, \n        p.MesEntrada, \n        DATE_DIFF(\'month\', p.MesEntrada, DATE_TRUNC(\'month\', r.TransactionDate)) AS MesRelativo, \n        r.TotalAmount\n    FROM Retail_Transaction_Clean r\n    JOIN PrimeiraCompra p USING (CustomerID)\n), \nResumo AS (\n    SELECT\n        MesEntrada, \n        MesRelativo, \n        SUM(TotalAmount) AS ReceitaMes\n    FROM ComprasComCoorte\n    WHERE MesEntrada IN (\'2023-04-01\', \'2023-10-01\')\n    GROUP BY MesEntrada, MesRelativo\n)\nSELECT\n    MesEntrada, \n    MesRelativo, \n    ReceitaMes, \n    SUM(ReceitaMes) OVER (\n        PARTITION BY MesEntrada\n        ORDER BY MesRelativo\n        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW\n    ) AS ReceitaAcumulada\nFROM Resumo\nORDER BY MesEntrada, MesRelativo;',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_28

,MesEntrada,MesRelativo,ReceitaMes,ReceitaAcumulada
0,2023-04-01,0,6.084079e+04,6.084079e+04
1,2023-04-01,1,6.192800e+02,6.146007e+04
2,2023-04-01,2,6.505560e+01,6.152512e+04
3,2023-04-01,4,6.709938e+02,6.219612e+04
4,2023-04-01,5,4.194533e+02,6.261557e+04
5,2023-04-01,6,1.810117e+03,6.442569e+04
6,2023-04-01,7,1.019738e+03,6.544543e+04
7,2023-04-01,9,3.205397e+02,6.576597e+04
8,2023-04-01,10,6.152700e+02,6.638124e+04
9,2023-04-01,11,5.182131e+02,6.689945e+04


**Questão 5 - Análise textual [Editado em 2025-09-27T23:18:00]**

**5.1 a)** Foi utilizada a cláusula ILIKE com as opções '%ap.%' e '%apt%' para encontrar os endereços que são possíveis apartamentos.
**b)** Um total de 22.12% da coluna StoreLocation pode ser considerado de apartamentos. 

**5.2 a)** Através do regexp foram extraídas as siglas referentes aos estados localizados no StoreLocation.
**b)** A partir da lista indicada, foi utilizada a cláusula "CASE WHEN" para categorizar os estados dentro dos bins 'Insular', 'Associated', 'Military'e 'Continental'.
**c)** Como visualização foi criado um gráfico de barras, para melhor observar o faturamento por tipo de região.  A região Continental foi a que mais teve impacto. 

**5.3** Nesta última parte foi utilizado o CONCAT() com as colunas chaves e o DATE_TRUNC() para formar frases: "Cliente x comprou y unidades do produto a em v."


In [29]:
df_29 = _dntk.execute_sql(
  'SELECT *\nFROM Retail_Transaction_Clean\nWHERE StoreLocation ILIKE \'%ap.%\'\n    OR StoreLocation ILIKE \'%apt%\';',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_29

,CustomerID,ProductID,Quantity,Price,TransactionDate,PaymentMethod,StoreLocation,ProductCategory,'DiscountApplied(%%)',TotalAmount
0,843385,A,8,56.025164,2023-10-11 06:48:00,Debit Card,"489 Juan Loop Apt. 093\nNorth Brettville, WV 7...",Home Decor,DiscountApplied(%%),419.766052
1,839609,B,5,23.857981,2024-02-27 11:13:00,Credit Card,528 Justin Expressway Apt. 336\nCabreraborough...,Electronics,DiscountApplied(%%),96.977925
2,266491,C,8,98.792726,2023-09-25 04:38:00,Debit Card,"388 Matthew Lane Apt. 592\nWest Andreachester,...",Books,DiscountApplied(%%),678.311227
3,26863,B,3,39.003285,2023-12-27 15:34:00,Credit Card,"623 William Track Apt. 920\nPort Dave, NE 02045",Electronics,DiscountApplied(%%),100.197369
4,798425,D,5,95.011762,2023-10-27 17:50:00,Cash,"1573 Petty Parkway Apt. 835\nJordanmouth, IN 0...",Clothing,DiscountApplied(%%),397.869376
...,...,...,...,...,...,...,...,...,...,...
22113,48854,B,5,10.737975,2024-03-14 15:36:00,Cash,"83119 Ian Square Apt. 868\nLake Mitchell, LA 7...",Home Decor,DiscountApplied(%%),50.863128
22114,460828,A,1,26.558805,2023-07-12 08:33:00,Debit Card,"8141 Ryan Springs Apt. 984\nLake Morgantown, V...",Electronics,DiscountApplied(%%),25.636783
22115,38578,D,9,71.253131,2023-07-31 01:31:00,Debit Card,"74267 Hurley Dam Apt. 760\nChaseborough, AR 58794",Home Decor,DiscountApplied(%%),640.767455
22116,530236,A,1,82.566022,2024-01-17 22:51:00,Debit Card,31050 Jeffrey Isle Apt. 494\nEast Charlesburgh...,Electronics,DiscountApplied(%%),66.401778


In [30]:
df_30 = _dntk.execute_sql(
  'SELECT\n    ROUND(100.0 * COUNT(*) FILTER (\n        WHERE StoreLocation ILIKE \'%ap.%\'\n            OR StoreLocation ILIKE \'%apt%\'\n    ) / COUNT(*), 2) AS PercentualApts\nFROM Retail_Transaction_Clean;',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_30

,PercentualApts
0,22.12


In [31]:
df_31 = _dntk.execute_sql(
  'SELECT\n    StoreLocation,\n    REGEXP_EXTRACT(StoreLocation, \'[ ,]([A-Z]{2})[ ]?[0-9]{5}\', 1) AS EstadoAbrev\nFROM Retail_Transaction_Clean\nLIMIT 20;',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_31

,StoreLocation,EstadoAbrev
0,"176 Andrew Cliffs\nBaileyfort, HI 93354",HI
1,"11635 William Well Suite 809\nEast Kara, MT 19483",MT
2,"910 Mendez Ville Suite 909\nPort Lauraland, MO...",MO
3,"87522 Sharon Corners Suite 500\nLake Tammy, MO...",MO
4,"0070 Michelle Island Suite 143\nHoland, VA 80142",VA
5,"8492 Jonathan Drive\nNorth Robertshire, TN 67532",TN
6,USNV Harrell\nFPO AA 62814,AA
7,"489 Juan Loop Apt. 093\nNorth Brettville, WV 7...",WV
8,528 Justin Expressway Apt. 336\nCabreraborough...,SD
9,"189 Wright Mews\nMartinfurt, MO 75932",MO


In [32]:
df_32 = _dntk.execute_sql(
  'WITH Estados AS (\n    SELECT\n        StoreLocation,\n        REGEXP_EXTRACT(StoreLocation, \'[ ,]([A-Z]{2})[ ]?[0-9]{5}\', 1) AS EstadoAbrev\n    FROM Retail_Transaction_Clean\n)\nSELECT\n    StoreLocation,\n    EstadoAbrev,\n    CASE\n        WHEN EstadoAbrev IN (\'PR\',\'GU\',\'VI\',\'AS\',\'MP\') THEN \'Insular\'\n        WHEN EstadoAbrev IN (\'FM\',\'MH\',\'PW\') THEN \'Associated\'\n        WHEN EstadoAbrev IN (\'AE\',\'AP\',\'AA\') THEN \'Military\'\n        ELSE \'Continental\'\n    END AS TipoRegiao\nFROM Estados\nLIMIT 20;',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_32

,StoreLocation,EstadoAbrev,TipoRegiao
0,"176 Andrew Cliffs\nBaileyfort, HI 93354",HI,Continental
1,"11635 William Well Suite 809\nEast Kara, MT 19483",MT,Continental
2,"910 Mendez Ville Suite 909\nPort Lauraland, MO...",MO,Continental
3,"87522 Sharon Corners Suite 500\nLake Tammy, MO...",MO,Continental
4,"0070 Michelle Island Suite 143\nHoland, VA 80142",VA,Continental
5,"8492 Jonathan Drive\nNorth Robertshire, TN 67532",TN,Continental
6,USNV Harrell\nFPO AA 62814,AA,Military
7,"489 Juan Loop Apt. 093\nNorth Brettville, WV 7...",WV,Continental
8,528 Justin Expressway Apt. 336\nCabreraborough...,SD,Continental
9,"189 Wright Mews\nMartinfurt, MO 75932",MO,Continental


In [33]:
df_33 = _dntk.execute_sql(
  'WITH Estados AS (\n    SELECT\n        StoreLocation,\n        REGEXP_EXTRACT(StoreLocation, \'[ ,]([A-Z]{2})[ ]?[0-9]{5}\', 1) AS EstadoAbrev\n    FROM Retail_Transaction_Clean\n),\nClassificados AS (\n  SELECT\n    StoreLocation,\n    EstadoAbrev,\n    CASE\n      WHEN EstadoAbrev IN (\'PR\',\'GU\',\'VI\',\'AS\',\'MP\') THEN \'Insular\'\n      WHEN EstadoAbrev IN (\'FM\',\'MH\',\'PW\')           THEN \'Associated\'\n      WHEN EstadoAbrev IN (\'AE\',\'AP\',\'AA\')           THEN \'Military\'\n      WHEN EstadoAbrev IS NULL                       THEN \'Indefinido\'\n      ELSE \'Continental\'\n    END AS TipoRegiao,\n    TotalAmount\n  FROM Estados e\n  JOIN Retail_Transaction_Clean r USING (StoreLocation)\n)\nSELECT\n  TipoRegiao,\n  COUNT(*) AS TotalPedidos,\n  SUM(TotalAmount) AS ReceitaTotal,\n  ROUND(AVG(TotalAmount),2) AS TicketMedio\nFROM Classificados\nGROUP BY TipoRegiao\nORDER BY ReceitaTotal DESC;',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_33

,TipoRegiao,TotalPedidos,ReceitaTotal,TicketMedio
0,Continental,77004,1.912163e+07,248.32
1,Military,10806,2.693719e+06,249.28
2,Insular,7528,1.847357e+06,245.40
3,Associated,4662,1.170787e+06,251.13


In [34]:
_dntk.DeepnoteChart(df_33, """{"layer":[{"layer":[{"layer":[{"mark":{"clip":true,"type":"bar","color":"#2266D3","tooltip":true},"encoding":{"x":{"sort":null,"type":"nominal","field":"TipoRegiao","scale":{"type":"linear"}},"y":{"axis":{"format":{"type":"default","decimals":null},"formatType":"numberFormatFromNumberType"},"type":"quantitative","field":"ReceitaTotal","scale":{"type":"linear"},"format":{"type":"default","decimals":null},"aggregate":"sum","formatType":"numberFormatFromNumberType"},"color":{"type":"nominal","datum":"ReceitaTotal","scale":{"range":["#2266D3"],"domain":["ReceitaTotal"]}},"xOffset":{"datum":"series_0"}},"transform":[]}]}],"resolve":{"scale":{"color":"independent"}}}],"title":"","config":{"legend":{"disable":false}},"$schema":"https://vega.github.io/schema/vega-lite/v5.json","encoding":{},"usermeta":{"seriesNames":["ReceitaTotal"],"seriesOrder":[0],"specSchemaVersion":2,"tooltipDefaultMode":true}}""", attach_selection=True, filters='[]')

In [35]:
df_34 = _dntk.execute_sql(
  'SELECT\n    CONCAT(\n        \'Cliente \', CustomerID,\n        \' comprou \', Quantity, \' unidades do produto \', ProductID,\n        \' em \', DATE_TRUNC(\'day\', TransactionDate)\n    ) AS FrasePedido\nFROM Retail_Transaction_Clean\nLIMIT 10;',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_34

,FrasePedido
0,Cliente 109318 comprou 7 unidades do produto C...
1,Cliente 993229 comprou 4 unidades do produto C...
2,Cliente 579675 comprou 8 unidades do produto A...
3,Cliente 799826 comprou 5 unidades do produto D...
4,Cliente 121413 comprou 7 unidades do produto A...
5,Cliente 463050 comprou 3 unidades do produto D...
6,Cliente 888163 comprou 7 unidades do produto D...
7,Cliente 843385 comprou 8 unidades do produto A...
8,Cliente 839609 comprou 5 unidades do produto B...
9,Cliente 184135 comprou 4 unidades do produto D...


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=b50e4344-4647-4a93-b699-42e32f41625a' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>